In [1]:
%pip install syncode==0.4.16 transformers==4.53.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.1/206.1 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 16.7 MB/s eta 0:00:00
  Created wheel for syncode: filename=syncode-0.4.16-py3-none-any.whl size=237009 sha256=caba079bad1ca9645998c43b224e7347a067cc9023b51f7018da5ee350dc5188
  Stored in directory: /root/.cache/pip/wheels/3f/a5/9b/9f6e1723cd27ff

In [2]:
import torch
import transformers
import syncode

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model.eval()

print("Model device:", model.device)
print("Model dtype:", model.dtype)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

[2026-10-25 04:25:25,881-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model device: cuda:0
Model dtype: torch.float16


In [4]:
import textwrap

grammar = r"""
?start: object

object: "{" ws "\"operands\"" ws ":" ws array ws "," ws "\"operator\"" ws ":" ws op_string ws "}"

array: "[" ws number ws "," ws number ws "]"

op_string: "\"+\"" | "\"-\"" | "\"*\"" | "\"/\""

number: SIGNED_NUMBER

ws: (" " | "\n" | "\t")*

%import common.SIGNED_NUMBER
"""

few_shot_prompt = textwrap.dedent("""
Extract the two floating point operands and the operator from the query.
Return only a JSON object with keys "operands" and "operator".
Do not compute the final answer.

Query: What is 327. multiplied by 11.0?
Output: {{"operands":[327.0,11.0],"operator":"*"}}

Query: What is 45.1 plus 23.54?
Output: {{"operands":[45.1,23.54],"operator":"+"}}

Query: What is 120.4 divided by 4.0?
Output: {{"operands":[120.4,4.0],"operator":"/"}}

Query: What is 19.75 minus 2.5?
Output: {{"operands":[19.75,2.5],"operator":"-"}}

Query: {query}
Output:
""").strip()

test_query = "What is 8.5 multiplied by 2.0?"
print(few_shot_prompt.format(query=test_query))

Extract the two floating point operands and the operator from the query.
Return only a JSON object with keys "operands" and "operator".
Do not compute the final answer.

Query: What is 327. multiplied by 11.0?
Output: {"operands":[327.0,11.0],"operator":"*"}

Query: What is 45.1 plus 23.54?
Output: {"operands":[45.1,23.54],"operator":"+"}

Query: What is 120.4 divided by 4.0?
Output: {"operands":[120.4,4.0],"operator":"/"}

Query: What is 19.75 minus 2.5?
Output: {"operands":[19.75,2.5],"operator":"-"}

Query: What is 8.5 multiplied by 2.0?
Output:


In [5]:
from syncode import Grammar, SyncodeLogitsProcessor

syncode_processor = SyncodeLogitsProcessor(grammar=Grammar(grammar), tokenizer=tokenizer, parse_output_only=True)

[2026-10-25 04:26:10,750-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_mask_6409611135_50257.pkl exist: False
[2026-10-25 04:26:10,750-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_mask_6409611135_50257.pkl.
[2026-10-25 04:26:11,270-syncode.mask_store.fsm_set] - 16 FSMs with 79 states initialized in 0.01 seconds
[2026-10-25 04:26:11,271-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-25 04:26:11,276-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 256 to 99


100%|██████████| 79/79 [00:20<00:00,  3.81it/s]


[2026-10-25 04:26:32,225-syncode.mask_store.mask_store] - Time taken to create mask store: 21.46 seconds


In [6]:
import json
import operator

def run_syncode(textual_query):
    inputs = tokenizer(few_shot_prompt.format(query=textual_query), return_tensors="pt").to(model.device)
    syncode_processor.reset()

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id, logits_processor=[syncode_processor])

    generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    data = json.loads(generated_text)

    operands = [float(value) for value in data["operands"]]
    op = data["operator"]

    operations = {
        "+": operator.add,
        "-": operator.sub,
        "*": operator.mul,
        "/": operator.truediv
    }

    if len(operands) != 2:
        raise ValueError("Number of operands should be 2")

    if op not in operations:
        raise ValueError("Unsupported operator")

    ans = float(operations[op](operands[0], operands[1]))

    return (operands, op, ans)


In [7]:
test_cases = [
    ("What is 73.25 plus 18.6?", [73.25, 18.6], "+", 91.85),
    ("What is 92.8 minus 37.45?", [92.8, 37.45], "-", 55.35),
    ("What is 6.75 multiplied by 12.4?", [6.75, 12.4], "*", 83.7),
    ("What is 144.0 divided by 12.0?", [144.0, 12.0], "/", 12.0),
    ("What is 5.0 minus 12.5?", [5.0, 12.5], "-", -7.5),
    ("What is -3.5 plus 8.0?", [-3.5, 8.0], "+", 4.5),
    ("What is 0.0 multiplied by 25.5?", [0.0, 25.5], "*", 0.0),
    ("What is 7.0 divided by 2.0?", [7.0, 2.0], "/", 3.5)
]

for query, expected_operands, expected_op, expected_ans in test_cases:
    print(query)
    try:
        operands, op, ans = run_syncode(query)
        passed = (operands == expected_operands and op == expected_op and abs(ans - expected_ans) < 1e-6)

        print("Output:", (operands, op, ans))
        print("Result:", "PASS" if passed else "FAIL")

    except Exception as e:
        print("Result: ERROR")
        print("Error:", str(e))


What is 73.25 plus 18.6?
Output: ([73.25, 18.6], '+', 91.85)
Result: PASS
What is 92.8 minus 37.45?
Output: ([92.8, 37.45], '-', 55.349999999999994)
Result: PASS
What is 6.75 multiplied by 12.4?
Output: ([6.75, 12.4], '*', 83.7)
Result: PASS
What is 144.0 divided by 12.0?
Output: ([144.0, 12.0], '/', 12.0)
Result: PASS
What is 5.0 minus 12.5?
Output: ([5.0, 12.5], '-', -7.5)
Result: PASS
What is -3.5 plus 8.0?
Output: ([-3.5, 8.0], '+', 4.5)
Result: PASS
What is 0.0 multiplied by 25.5?
Output: ([0.0, 25.5], '*', 0.0)
Result: PASS
What is 7.0 divided by 2.0?
Output: ([7.0, 2.0], '/', 3.5)
Result: PASS
